In [11]:
import ollama
import httpx
from loguru import logger
from IPython.display import JSON
import json

In [12]:
llm_model = "gpt-oss:20b"

In [13]:
from pydantic import BaseModel, Field

class Answer(BaseModel):
    answer: str = Field(..., description="The main answer generated by the model.")
    answer_confidence_score: float = Field(
        ..., ge=0.0, le=1.0, description="Confidence score of the answer, between 0 and 1."
    )

In [14]:
def wikipedia(country: str) -> str:
    """"
    Wikipedia tool function that simulates fetching information about a country from Wikipedia.

    Args:
        country (str): The name of the country to query.
    
    Returns:
        str: A string containing the information retrieved from Wikipedia about the specified country.
    """
    try:
        capitals = {
            'Italy': 'Rome',
            'Spain': 'Madrid',
        }
        capital = capitals.get(country, "Capital not found.")
        return capital
    except Exception as e:
        logger.error(f"Error calling Wikipedia API: {e}")
        return "Error fetching data from Wikipedia."

In [ ]:
class Agent:
    def __init__(self, system=""):
        self.system = system
        self.messages = []
        self.tools_used = []  # Initialize a list to track tools used
        if self.system:
            self.messages.append({"role": "system", "content": system})

    def __call__(self, message):
        try:
            self.messages.append({"role": "user", "content": message})
            result = self.execute()
            self.messages.append({"role": "assistant", "content": result["result"]["answer"]})
            return result
        except Exception as e:
            logger.error(f"Error during agent call: {e}")
            return "An error occurred while processing your request."

    def execute(self):
        list_tool_calls = []
        try:
            response = ollama.chat(
                    model=llm_model,
                    messages=self.messages,
                    options={"temperature": 0.01},
                    tools=[wikipedia],
                    format=Answer.model_json_schema(),
                )

            tool_calls = response["message"].get("tool_calls", [])
            if tool_calls:
                for call in tool_calls:
                    if call.function.name == 'wikipedia':
                        result = wikipedia(call.function.arguments['country'])
                    else:
                        result = 'Unknown tool'
                    self.messages.append({'role': 'tool', 
                                    'tool_name': call.function.name, 
                                    'content': str(result)})
                    
                    list_tool_calls.append({
                        'tool_name': call.function.name,
                        'arguments': call.function.arguments
                    }) 
                final_response = ollama.chat(
                        model=llm_model,
                        messages=self.messages,
                        options={"temperature": 0.01},
                        tools=[wikipedia],
                        format=Answer.model_json_schema()

                    )
                answer = final_response["message"]["content"]
                return {"result": json.loads(answer),
                        "tool_calls": list_tool_calls}
            return {"result": json.loads(response["message"]["content"]),
                    "tool_calls": list_tool_calls}
        except Exception as e:
            logger.error(f"Error during agent execution: {e}")
            return "An error occurred while processing your request."



In [16]:
# Example usage
agent = Agent(system="You are a helpful assistant. You can use tools to answer questions.")

In [17]:
response = agent(message="What is the capital of Italy?")
response

{'result': {'answer': 'Rome is the capital of Italy.',
  'answer_confidence_score': 1.0},
 'tool_calls': []}

In [18]:
response = agent(message="And for Spain using the fake Wikipedia?")
response

{'result': {'answer': 'Madrid', 'answer_confidence_score': 1.0},
 'tool_calls': [{'tool_name': 'wikipedia', 'arguments': {'country': 'Spain'}}]}

In [19]:
response = agent(message="Which tools did you use to answer the previous question?")
response

{'result': {'answer': 'I used the Wikipedia tool to fetch the capital of Spain.',
  'answer_confidence_score': 1},
 'tool_calls': []}

In [20]:
response = agent(message="Thank you!")
response

{'result': {'answer': "You're welcome! If you have any more questions, feel free to ask.",
  'answer_confidence_score': 0.99},
 'tool_calls': []}